In [ ]:
import csv
import os
import math
import time
from pathlib import Path
import gurobipy
import numpy as np
import scipy.io
import torch
from torch import nn
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from scipy.stats import multivariate_normal

In [ ]:
METHODS = ["GMM", "IND", "NW", "NN"]

BASE_SEED = 42
N = 39
NUM_DAYS = 8
T = NUM_DAYS - 1  
QUARTERS = [1,2,3,4]
VAL_SIZE = 9
IN_SAMPLE_CV_TRAIN_N = N - VAL_SIZE
MAX_ITER = 100
EPS_CV_MAX_ITER = 30
NUM_ITER = MAX_ITER 
RUN_OOS = True
VERBOSE = False

EPS_LIST = [0.001, 0.005, 0.01, 0.05]

# GMM transition estimator; used only by method == "GMM"
GMM_MIN_COMPONENTS = 1
GMM_MAX_COMPONENTS = 3
GMM_COMPONENT_SELECTION = "aic"
GMM_COVARIANCE_TYPE = "full"
GMM_LOG_MARG_FLOOR = -20

NW_BANDWIDTH_MULTIPLIERS = [1, 5, 10]

USE_NEURAL_WARM_START = True
NUM_NEURAL_CUTS = 16 
NEURAL_HIDDEN = 128
NEURAL_EPOCHS = 500
NEURAL_LR = 1e-3
NEURAL_TEACHER_ITER = NUM_ITER
NEURAL_REFINEMENT_ITER = 0

# PyTorch uses GPU only for neural cut training/prediction when available.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", DEVICE)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Storage and dispatch constants
K = 3
H = 24
cap = np.asarray([5000.0, 2000.0, 1000.0], dtype=float)
rho = np.asarray([0.98, 0.99, 0.995], dtype=float)
rho_c = np.asarray([0.8, 0.9, 1.0], dtype=float)
rho_d = np.asarray([0.8, 0.9, 1.0], dtype=float)
INIT_LEVEL = np.asarray([500.0, 500.0, 500.0], dtype=float)
LB = -gurobipy.GRB.INFINITY

OutputFlag = 0
np.random.seed(BASE_SEED)

In [ ]:
# Data Loading
def find_wind_data_dir():
    env_dir = os.environ.get('WIND_DATA_DIR')
    candidates = []
    if env_dir:
        candidates.append(Path(env_dir).expanduser())
    candidates.extend([
        Path.cwd() / 'data' / 'wind_data',
        Path.cwd() / 'data',
        Path.cwd() / 'wind_data',
        Path.cwd(),
        Path.cwd().parent / 'data' / 'wind_data',
        Path.cwd().parent / 'data',
        Path.cwd().parent / 'wind_data',
    ])
    for d in candidates:
        if (d / 'OH_all.mat').exists() and (d / 'price_all.mat').exists():
            return d
    searched = '\n'.join(str(d) for d in candidates)
    raise FileNotFoundError(
        'Could not find OH_all.mat and price_all.mat. Place both files in '
        'data/wind_data, data, wind_data, the notebook working directory, '
        'or set WIND_DATA_DIR.\nSearched:\n' + searched
    )

def split_daily(arr, T_):
    if arr is None:
        return None
    arr = np.asarray(arr, dtype=float)
    required_hours = (T_ + 1) * H
    if required_hours != NUM_DAYS * H:
        raise ValueError(
            f'Horizon mismatch: T + 1 = {T_ + 1} daily states, but NUM_DAYS={NUM_DAYS}.'
        )
    if arr.ndim != 2:
        raise ValueError(f'Expected a 2D trajectory-by-hour array, got shape {arr.shape}.')
    if arr.shape[1] < required_hours:
        raise ValueError(
            f'Each trajectory needs at least {required_hours} hourly entries '
            f'({NUM_DAYS} days * {H} hours), but got {arr.shape[1]}.'
        )
    arr = arr[:, :required_hours]
    daily = np.zeros((T_ + 1, arr.shape[0], H), dtype=float)
    for t in range(T_ + 1):
        daily[t] = arr[:, t * H:(t + 1) * H]
    return daily

In [ ]:
# Kernal functions
def normalize_rows(mat):
    q = np.asarray(mat, dtype=float).copy()
    q[~np.isfinite(q)] = 0.0
    q = np.maximum(q, 0.0)
    row_sum = q.sum(axis=-1, keepdims=True)
    bad = row_sum.squeeze(axis=-1) <= 0.0
    if np.any(~bad):
        q[~bad] = q[~bad] / row_sum[~bad]
    if np.any(bad):
        q[bad] = 1.0 / q.shape[-1]
    return q

def normalize_probability_rows(mat):
    q = np.asarray(mat, dtype=float).copy()
    q[~np.isfinite(q)] = 0.0
    q = np.maximum(q, 0.0)
    row_sum = q.sum(axis=-1, keepdims=True)
    bad = row_sum.squeeze(axis=-1) <= 0.0
    if np.any(~bad):
        q[~bad] = q[~bad] / row_sum[~bad]
    if np.any(bad):
        q[bad] = 1.0 / q.shape[-1]
    return q

def get_kernel_bandwidth(samples):
    N_samples, K_dim = samples.shape
    S = np.cov(samples.T, ddof=1)
    S = np.asarray(S, dtype=float)
    S = S + 1e-8 * np.eye(K_dim)
    H_band = (1.0 / ((2 * K_dim + 1) * N_samples)) ** (2.0 / (K_dim + 4))
    return H_band, S

def get_nw_bandwidth_candidates():
    raw = globals().get("NW_BANDWIDTH_MULTIPLIERS", [1.0])
    if raw is None:
        raw = [1.0]
    vals = []
    for value in np.asarray(raw, dtype=float).reshape(-1):
        value = float(value)
        if np.isfinite(value) and value > 0.0:
            vals.append(value)
    if not vals:
        vals = [1.0]
    out = []
    for value in vals:
        if not any(abs(value - old) <= 1e-12 for old in out):
            out.append(value)
    return out

def transition_diagnostics(name, transition_matrix):
    print(f'[{name}] transition diagnostic')
    for t, mat in enumerate(transition_matrix):
        q = np.asarray(mat, dtype=float)
        if q.ndim == 1:
            q = q.reshape(1, -1)
        ess = 1.0 / np.sum(q * q, axis=1)
        if q.shape[0] == 1:
            print(f't={t:02d} shape={q.shape} sum={q.sum():.6f} min={q.min():.3e} max={q.max():.3e} ESS={ess[0]:.2f}')
        else:
            row_err = float(np.max(np.abs(q.sum(axis=1) - 1.0)))
            print(f't={t:02d} shape={q.shape} row_err={row_err:.2e} min={q.min():.3e} max={q.max():.3e} '
                  f'ESS_mean={ess.mean():.2f} ESS_min={ess.min():.2f} ESS_max={ess.max():.2f}')

In [ ]:
# NW transition
def get_nw_exp_kernel_bandwidth(samples):
    N_samples, K_dim = samples.shape
    S = np.cov(samples.T, ddof=1)
    S = np.asarray(S, dtype=float)
    S = S + 1e-8 * np.eye(K_dim)
    H_band = (1.0 / ((2 * K_dim + 1) * N_samples)) ** (2.0 / (K_dim + 4))
    return H_band, S

def get_nw_exp_kernel(x, y, H_band, S):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    diff = x - y
    X = np.linalg.solve(S, diff)
    quad = float(np.dot(X.T, diff))
    scaled_quad = max(quad / max(float(H_band), 1e-12), 0.0)
    return float(np.exp(-0.5 * np.sqrt(scaled_quad)))

def build_power_price_nw_transition_matrix(daily_power, daily_price, bandwidth=1.0):
    S_count, n, power_dim = daily_power.shape
    _, _, price_dim = daily_price.shape

    power_pca_dim = min(3, n, power_dim)
    price_pca_dim = min(3, n, price_dim)
    bw_mult = float(bandwidth)
    if not np.isfinite(bw_mult) or bw_mult <= 0.0:
        raise ValueError(f'bandwidth multiplier must be positive, got {bandwidth}.')

    power_score = np.zeros((S_count, n, power_pca_dim), dtype=float)
    price_score = np.zeros((S_count, n, price_pca_dim), dtype=float)
    power_H = np.zeros(S_count, dtype=float)
    price_H = np.zeros(S_count, dtype=float)
    power_H_base = np.zeros(S_count, dtype=float)
    price_H_base = np.zeros(S_count, dtype=float)
    power_S = np.zeros((S_count, power_pca_dim, power_pca_dim), dtype=float)
    price_S = np.zeros((S_count, price_pca_dim, price_pca_dim), dtype=float)
    power_pca = []
    price_pca = []

    for t in range(S_count):
        pca_power = PCA(n_components=power_pca_dim, random_state=42)
        pca_price = PCA(n_components=price_pca_dim, random_state=42)
        power_score[t] = pca_power.fit_transform(daily_power[t])
        price_score[t] = pca_price.fit_transform(daily_price[t])
        power_H_base[t], power_S[t] = get_nw_exp_kernel_bandwidth(power_score[t])
        price_H_base[t], price_S[t] = get_nw_exp_kernel_bandwidth(price_score[t])
        power_H[t] = bw_mult * power_H_base[t]
        price_H[t] = bw_mult * price_H_base[t]
        power_pca.append(pca_power)
        price_pca.append(pca_price)

    weightNW = np.zeros((T, n, n), dtype=float)
    for t in range(T):
        for i in range(n):
            for j in range(n):
                weightNW[t, i, j] = (
                    get_nw_exp_kernel(
                        power_score[t, i, :],
                        power_score[t, j, :],
                        power_H[t],
                        power_S[t],
                    )
                    * get_nw_exp_kernel(
                        price_score[t, i, :],
                        price_score[t, j, :],
                        price_H[t],
                        price_S[t],
                    )
                )
            weightNW[t, i, :] = normalize_rows(weightNW[t, i, :].reshape(1, -1))[0]

    dep_transition_matrix = [np.ones((1, n), dtype=float) / n]
    for t in range(1, T + 1):
        dep_transition_matrix.append(weightNW[t - 1].copy())

    info = {
        'feature_name': 'POWER_PRICE',
        'nw_feature_mode': 'separate_power_price_product_pca',
        'nw_power_pca_dim': power_pca_dim,
        'nw_price_pca_dim': price_pca_dim,
        'nw_raw_power_dim': power_dim,
        'nw_raw_price_dim': price_dim,
        'nw_kernel': 'exp_mahalanobis_product',
        'nw_bandwidth_multiplier': bw_mult,
        'nw_bandwidth': bw_mult,
        'power_score': power_score,
        'price_score': price_score,
        'power_H': power_H,
        'price_H': price_H,
        'power_H_base': power_H_base,
        'price_H_base': price_H_base,
        'power_S': power_S,
        'price_S': price_S,
        'power_pca': power_pca,
        'price_pca': price_pca,
    }
    return dep_transition_matrix, info

def power_price_nw_oos_row(transition_info, stage_t, kk, daily_power_oos, daily_price_oos):
    pca_power = transition_info['power_pca'][stage_t]
    pca_price = transition_info['price_pca'][stage_t]
    power_raw = daily_power_oos[stage_t, kk].reshape(1, -1)
    price_raw = daily_price_oos[stage_t, kk].reshape(1, -1)
    power_oos_score = pca_power.transform(power_raw)[0]
    price_oos_score = pca_price.transform(price_raw)[0]

    weights = np.zeros(N, dtype=float)
    for n in range(N):
        weights[n] = (
            get_nw_exp_kernel(
                transition_info['power_score'][stage_t, n, :],
                power_oos_score,
                transition_info['power_H'][stage_t],
                transition_info['power_S'][stage_t],
            )
            * get_nw_exp_kernel(
                transition_info['price_score'][stage_t, n, :],
                price_oos_score,
                transition_info['price_H'][stage_t],
                transition_info['price_S'][stage_t],
            )
        )
    return normalize_rows(weights.reshape(1, -1))[0]




In [ ]:
# GMM transition
def select_feature_raw(feature_name, daily_wind, daily_power, daily_price):
    name = feature_name.upper()
    if name == 'WIND_PRICE':
        if daily_wind is None:
            raise ValueError('feature_name=WIND_PRICE requires weekly_wind in NC_all.mat.')
        return np.concatenate([daily_wind, daily_price], axis=2)
    if name == 'POWER_PRICE':
        return np.concatenate([daily_power, daily_price], axis=2)
    if name == 'WIND_ONLY':
        if daily_wind is None:
            raise ValueError('feature_name=WIND_ONLY requires weekly_wind in NC_all.mat.')
        return daily_wind.copy()
    if name == 'POWER_ONLY':
        return daily_power.copy()
    raise ValueError(f'Unknown feature_name={feature_name!r}.')

def fit_stage_feature_preprocessor(feature_raw, feature_name='POWER_PRICE'):
    feature_raw = np.asarray(feature_raw, dtype=float)
    if feature_raw.ndim != 3:
        raise ValueError(f'feature_raw must be 3D, got shape {feature_raw.shape}.')
    _, _, d = feature_raw.shape
    return feature_raw.copy(), {
        'mode': 'raw_power_price_features',
        'feature_name': str(feature_name).upper(),
        'raw_dim': d,
    }

def transform_stage_feature(feature_raw, preprocessor):
    feature_raw = np.asarray(feature_raw, dtype=float)
    if feature_raw.ndim != 3:
        raise ValueError(f'feature_raw must be 3D, got shape {feature_raw.shape}.')
    raw_dim = int(preprocessor['raw_dim'])
    if feature_raw.shape[2] != raw_dim:
        raise ValueError(
            f'OOS feature dimension mismatch: got d={feature_raw.shape[2]}, expected {raw_dim}.'
        )
    return feature_raw.copy()

def gmm_reg_covar_from_nw_bandwidth(nw_bandwidth, nw_covariance):
    S = np.asarray(nw_covariance, dtype=float)
    if S.ndim != 2 or S.shape[0] != S.shape[1]:
        raise ValueError(f'nw_covariance must be a square matrix, got shape {S.shape}.')
    diag_s = np.diag(S)
    diag_s = diag_s[np.isfinite(diag_s) & (diag_s > 0.0)]
    max_sii = float(np.max(diag_s)) if diag_s.size else 1.0
    reg_covar = max_sii
    return max(float(reg_covar), 1e-12), max_sii

def fit_gmm_pair_model(pair_samples, nw_bandwidth, nw_covariance):
    X = np.asarray(pair_samples, dtype=float)
    if X.ndim != 2:
        raise ValueError(f'GMM pair samples must be 2D, got shape {X.shape}.')

    max_k = int(globals().get('GMM_MAX_COMPONENTS', 3))
    min_k = int(globals().get('GMM_MIN_COMPONENTS', 1))
    max_k = max(1, min(max_k, X.shape[0]))
    min_k = max(1, min(min_k, max_k))
    k_candidates = list(range(min_k, max_k + 1))
    reg_covar, nw_cov_max_diag = gmm_reg_covar_from_nw_bandwidth(nw_bandwidth, nw_covariance)

    fitted = []
    aic_records = []
    for k in k_candidates:
        try:
            gmm = GaussianMixture(
                n_components=int(k),
                covariance_type=GMM_COVARIANCE_TYPE,
                reg_covar=float(reg_covar),
                random_state=BASE_SEED,
                n_init=30,
            )
            gmm.fit(X)
            aic = float(gmm.aic(X))
            bic = float(gmm.bic(X))
        except Exception as exc:
            if VERBOSE:
                print(f'GMM fit failed for k={k}, reg_covar={reg_covar:g}: {exc}')
            continue

        aic_records.append({
            'k': int(k),
            'reg_covar': float(reg_covar),
            'nw_bandwidth': float(nw_bandwidth),
            'nw_cov_max_diag': float(nw_cov_max_diag),
            'aic': aic,
            'bic': bic,
        })
        fitted.append((aic, bic, int(k), gmm))

    if not fitted:
        raise RuntimeError(
            f'All GMM fits failed for k_candidates={k_candidates} and reg_covar={reg_covar:g}.'
        )

    criterion = str(globals().get('GMM_COMPONENT_SELECTION', 'aic')).lower()
    if criterion == 'aic':
        _, _, selected_gmm_K, best_gmm = min(fitted, key=lambda item: item[0])
    elif criterion == 'bic':
        _, _, selected_gmm_K, best_gmm = min(fitted, key=lambda item: item[1])
    else:
        raise ValueError(f'Unsupported GMM_COMPONENT_SELECTION={GMM_COMPONENT_SELECTION!r}.')

    return best_gmm, selected_gmm_K, float(reg_covar), aic_records

def gmm_conditional_row(gmm, x_prev, candidates):
    x_prev = np.asarray(x_prev, dtype=float).reshape(1, -1)
    candidates = np.asarray(candidates, dtype=float)
    if candidates.ndim != 2:
        raise ValueError(f'candidates must be 2D, got shape {candidates.shape}.')

    joint = np.hstack([np.repeat(x_prev, candidates.shape[0], axis=0), candidates])
    if joint.shape[1] != gmm.means_.shape[1]:
        raise ValueError(
            f'GMM dimension mismatch: joint dim={joint.shape[1]} but GMM dim={gmm.means_.shape[1]}.'
        )
    log_joint = gmm.score_samples(joint)
    d_prev = x_prev.shape[1]

    log_terms = []
    for k in range(gmm.n_components):
        mean = gmm.means_[k, d_prev:]
        if gmm.covariance_type == "diag":
            var = np.asarray(gmm.covariances_[k, d_prev:], dtype=float)
            var = np.maximum(var, 1e-12)
            diff = candidates - mean
            log_pdf = -0.5 * (
                np.sum((diff * diff) / var, axis=1)
                + np.sum(np.log(2.0 * np.pi * var))
            )
        elif gmm.covariance_type == "full":
            cov = gmm.covariances_[k][d_prev:, d_prev:]
            log_pdf = multivariate_normal.logpdf(candidates, mean=mean, cov=cov, allow_singular=True)
        else:
            raise ValueError(f'Unsupported covariance_type={gmm.covariance_type!r}.')
        log_terms.append(np.log(gmm.weights_[k]) + log_pdf)

    log_marg_next = logsumexp(np.vstack(log_terms), axis=0)
    log_marg_floor = globals().get("GMM_LOG_MARG_FLOOR", None)
    if log_marg_floor is not None:
        log_marg_next = np.maximum(log_marg_next, float(log_marg_floor))

    return log_joint - log_marg_next

def gmm_likelihood_ratio_weights(log_scores):
    log_scores = np.asarray(log_scores, dtype=float)
    log_scores = np.where(np.isfinite(log_scores), log_scores, -np.inf)
    log_norm = logsumexp(log_scores, axis=-1, keepdims=True)

    if np.any(~np.isfinite(log_norm)):
        raise ValueError(
            "All GMM likelihood-ratio log-scores are non-finite; "
            "cannot compute likelihood-ratio weights."
        )

    return np.exp(log_scores - log_norm)

def build_gmm_transition_matrix(raw_feature, feature_name='POWER_PRICE'):
    feature_states, preprocessor = fit_stage_feature_preprocessor(raw_feature, feature_name=feature_name)
    S, n, _ = feature_states.shape

    transition = [np.ones((1, n), dtype=float) / n]
    gmm_by_stage = []
    selected_gmm_K_by_stage = []
    selected_gmm_reg_covar_by_stage = []
    selected_gmm_bandwidth_by_stage = []
    selected_gmm_cov_max_diag_by_stage = []
    gmm_aic_records_by_stage = []

    for t in range(1, S):
        pair_samples = np.concatenate([feature_states[t - 1], feature_states[t]], axis=1)
        nw_bandwidth, nw_covariance = get_kernel_bandwidth(feature_states[t - 1])
        gmm, selected_gmm_K, selected_reg_covar, aic_records = fit_gmm_pair_model(pair_samples, nw_bandwidth, nw_covariance)
        gmm_by_stage.append(gmm)
        selected_gmm_K_by_stage.append(int(selected_gmm_K))
        selected_gmm_reg_covar_by_stage.append(float(selected_reg_covar))
        nw_cov_max_diag = float(np.max(np.diag(nw_covariance)))
        selected_gmm_bandwidth_by_stage.append(float(nw_bandwidth))
        selected_gmm_cov_max_diag_by_stage.append(float(nw_cov_max_diag))
        gmm_aic_records_by_stage.append(aic_records)

        if VERBOSE:
            print(f'GMM AIC K-selection fit for transition {t - 1}->{t}:')
            for rec in aic_records:
                print(
                    f"k={rec['k']} nw_bandwidth={rec['nw_bandwidth']:.6g} "
                    f"maxSii={rec['nw_cov_max_diag']:.6g} "
                    f"reg={rec['reg_covar']:.6g} "
                    f"AIC={rec['aic']:.6g} BIC={rec['bic']:.6g}"
                )
            print(
                f'selected_gmm_K={selected_gmm_K}, '
                f'nw_kernel=gaussian, '
                f'nw_bandwidth={nw_bandwidth:.6g}, '
                f'maxSii={nw_cov_max_diag:.6g}, '
                f'selected_reg_covar={selected_reg_covar:.6g}'
            )

        rows = []
        for i in range(n):
            rows.append(gmm_conditional_row(gmm, feature_states[t - 1, i], feature_states[t]))
        Q = gmm_likelihood_ratio_weights(np.vstack(rows))
        Q = normalize_rows(Q)
        transition.append(Q)

    transition_info = {
        'preprocessor': preprocessor,
        'feature_states': feature_states,
        'gmm_by_stage': gmm_by_stage,
        'selected_gmm_K_by_stage': selected_gmm_K_by_stage,
        'selected_gmm_reg_covar_by_stage': selected_gmm_reg_covar_by_stage,
        'selected_gmm_bandwidth_by_stage': selected_gmm_bandwidth_by_stage,
        'selected_gmm_cov_max_diag_by_stage': selected_gmm_cov_max_diag_by_stage,
        'selected_gmm_K': ','.join(str(k) for k in selected_gmm_K_by_stage),
        'selected_gmm_reg_covar': ','.join(f'{r:g}' for r in selected_gmm_reg_covar_by_stage),
        'selected_gmm_bandwidth': ','.join(f'{h:g}' for h in selected_gmm_bandwidth_by_stage),
        'selected_gmm_cov_max_diag': ','.join(f'{s:g}' for s in selected_gmm_cov_max_diag_by_stage),
        'gmm_aic_records_by_stage': gmm_aic_records_by_stage,
        'feature_name': feature_name,
    }
    return transition, transition_info

def build_oos_transition_cache(mode, daily_wind_oos, daily_power_oos, daily_price_oos, transition_info):
    if daily_power_oos is None or daily_power_oos.shape[1] == 0:
        return None

    mode_u = mode.upper()
    n_oos = int(daily_power_oos.shape[1])
    cache = np.zeros((T, n_oos, N), dtype=float)

    if mode_u == 'IND':
        cache[:] = np.ones(N, dtype=float) / N
        return cache

    if mode_u == 'NW':
        for t in range(T):
            for kk in range(n_oos):
                cache[t, kk] = power_price_nw_oos_row(
                    transition_info,
                    t,
                    kk,
                    daily_power_oos,
                    daily_price_oos,
                )
        return cache

    if mode_u == 'GMM':
        raw_oos = select_feature_raw(
            transition_info['feature_name'],
            daily_wind_oos,
            daily_power_oos,
            daily_price_oos,
        )
        feat_oos = transform_stage_feature(raw_oos, transition_info['preprocessor'])
        for t in range(T):
            gmm = transition_info['gmm_by_stage'][t]
            candidates = transition_info['feature_states'][t + 1]
            for kk in range(n_oos):
                row = gmm_conditional_row(gmm, feat_oos[t, kk], candidates)
                q = gmm_likelihood_ratio_weights(row.reshape(1, -1))
                q = normalize_rows(q)
                cache[t, kk] = q[0]
        return cache

    raise ValueError(mode)

In [ ]:
# SDDP model construction
def optimize_or_raise(model, context):
    model.update()
    model.optimize()
    ok = {gurobipy.GRB.OPTIMAL, gurobipy.GRB.SUBOPTIMAL}
    if model.Status not in ok:
        raise RuntimeError(f'{context}: Gurobi status {model.Status}, cannot read solution values.')
    return model

def set_storage_rhs(model, storage):
    for l in range(K):
        c = model.getConstrByName(f'grad_constr_{l}')
        if c is None:
            raise KeyError(f'Missing grad_constr_{l}')
        c.RHS = float(storage[l])

def update_oos_objective_and_weights(model, price_vec, weights, eps):
    price_vec = np.asarray(price_vec, dtype=float)
    weights = normalize_probability_rows(np.asarray(weights, dtype=float).reshape(1, -1))[0]
    sqrt_eps = math.sqrt(max(float(eps), 0.0))
    for h in range(H):
        model.getVarByName(f'x[{h}]').Obj = -float(price_vec[h])
    for n in range(N):
        for h in range(H):
            var = model.getVarByName(f'unmet[{n},{h}]')
            if var is not None:
                var.Obj = float(2.0 * weights[n] * price_vec[h])
    beta = model.getVarByName('beta')
    if beta is None:
        model.update()
        return
    beta.Obj = sqrt_eps
    for n in range(N):
        qn = float(weights[n])
        sqrt_q = math.sqrt(qn)
        phi = model.getVarByName(f'phi[{n}]')
        gamma = model.getVarByName('gamma')
        alpha = model.getVarByName(f'alpha[{n}]')
        mu = model.getVarByName(f'mu[{n}]')
        lmbda = model.getVarByName(f'lambda[{n}]')
        gamma_const = model.getConstrByName(f'gamma_const[{n}]')
        if phi is not None:
            phi.Obj = sqrt_eps
        if mu is not None:
            mu.Obj = sqrt_q
        if lmbda is not None:
            lmbda.Obj = -sqrt_q
        if (
            gamma_const is not None and gamma is not None and alpha is not None
            and mu is not None and lmbda is not None
        ):
            model.chgCoeff(gamma_const, gamma, sqrt_q)
            model.chgCoeff(gamma_const, alpha, -sqrt_q)
            model.chgCoeff(gamma_const, mu, 1.0)
            model.chgCoeff(gamma_const, lmbda, -1.0)
    model.update()

def add_alpha_cuts_to_stage(stage_models, t, iteration, obj_next, dual_next, x_ref,
                            transition_matrix=None, cut_records=None):
    x_ref = np.asarray(x_ref, dtype=float)
    obj_next = np.asarray(obj_next, dtype=float)
    dual_next = np.asarray(dual_next, dtype=float)

    if obj_next.shape[0] != N or dual_next.shape != (N, K):
        raise ValueError(
            f'Expected obj_next shape {(N,)} and dual_next shape {(N, K)}, '
            f'got {obj_next.shape} and {dual_next.shape}.'
        )

    child_records = []
    if cut_records is not None:
        for n in range(N):
            child_intercept = float(obj_next[n] - np.dot(dual_next[n], x_ref))
            child_records.append({
                'stage': int(t),
                'iteration': int(iteration),
                'node': -1,
                'child': int(n),
                'a': child_intercept,
                'b': np.asarray(dual_next[n], dtype=float).tolist(),
                'cut_type': 'multi',
            })

    for idx, model in enumerate(stage_models[t]):
        for n in range(N):
            alpha_var = model.getVarByName(f'alpha[{n}]')
            now_vars = [model.getVarByName(f'now[{n},{l}]') for l in range(K)]
            if alpha_var is None or any(v is None for v in now_vars):
                continue

            child_intercept = float(obj_next[n] - np.dot(dual_next[n], x_ref))
            expr = alpha_var - gurobipy.quicksum(
                float(dual_next[n, l]) * now_vars[l] for l in range(K)
            )
            model.addConstr(
                expr >= child_intercept,
                name=f'multicut_it{iteration}_t{t}_node{idx}_child{n}',
            )
        model.update()

    if cut_records is not None:
        cut_records.extend(child_records)

def build_stage_models(Markov_states_power, Markov_states_price, transition_matrix, eps):
    models = []
    for t in range(T):
        stage_models = []
        for idx in range(N):
            q = normalize_probability_rows(np.asarray(transition_matrix[t + 1][idx]).reshape(1, -1))[0]
            price = np.asarray(Markov_states_price[t][idx], dtype=float)
            m = gurobipy.Model(f'SAA_LP_{t+1}stage_{idx+1}instance.lp')
            m.Params.OutputFlag = OutputFlag
            m.Params.LogToConsole = 0
            m.Params.DualReductions = 0
            m.Params.Threads = 1
            m.Params.Seed = BASE_SEED

            now = m.addMVar((N, K), name='now')
            z = m.addVars(K, name='copy')
            x = m.addVars(H, obj=-price, name='x')
            e_c = m.addMVar((N, H), name='met')
            e_w = m.addMVar((N, H), name='wasted')

            e_u_coeff = np.zeros((N, H), dtype=float)
            for i in range(N):
                e_u_coeff[i] = q[i] * price
            e_u = m.addMVar((N, H), obj=2.0 * e_u_coeff, name='unmet')

            e_chg_0 = m.addMVar((N, H), ub=cap[0], name='charged_0')
            e_chg_1 = m.addMVar((N, H), ub=cap[1], name='charged_1')
            e_chg_2 = m.addMVar((N, H), ub=cap[2], name='charged_2')
            e_dis_0 = m.addMVar((N, H), ub=cap[0], name='discharged_0')
            e_dis_1 = m.addMVar((N, H), ub=cap[1], name='discharged_1')
            e_dis_2 = m.addMVar((N, H), ub=cap[2], name='discharged_2')
            now_next_0 = m.addMVar((N, H), ub=cap[0], name='level_0')
            now_next_1 = m.addMVar((N, H), ub=cap[1], name='level_1')
            now_next_2 = m.addMVar((N, H), ub=cap[2], name='level_2')

            if t < T - 1:
                gamma = m.addVar(obj=1.0, lb=LB, name='gamma')
                beta = m.addVar(obj=math.sqrt(max(float(eps), 0.0)), name='beta')
                phi = m.addVars(N, obj=math.sqrt(max(float(eps), 0.0)) * np.ones(N), name='phi')
                mu = m.addVars(N, obj=np.sqrt(q), name='mu')
                lmbda = m.addVars(N, obj=-np.sqrt(q), name='lambda')
                alpha = m.addVars(N, name='alpha', lb=-1e8)
                m.update()
            else:
                alpha = None
                m.update()

            for l in range(K):
                init_rhs = INIT_LEVEL[l] if t == 0 else 0.0
                m.addConstr(z[l] == float(init_rhs), name=f'grad_constr_{l}')

            for n in range(N):
                for h in range(H):
                    wind_expr = e_chg_0[n, h] + e_chg_1[n, h] + e_chg_2[n, h] + e_w[n, h] + e_c[n, h]
                    m.addConstr(wind_expr == float(Markov_states_power[t + 1][n][h]), name=f'wind_const_[{n}]_[{h}]')
                    dispatch_expr = e_dis_0[n, h] + e_dis_1[n, h] + e_dis_2[n, h] + e_u[n, h] + e_c[n, h]
                    m.addConstr(x[h] == dispatch_expr, name=f'dispatch_const_[{n}]_[{h}]')

                for h in range(H):
                    if h == 0:
                        m.addConstr(now_next_0[n, 0] == rho[0] * z[0] + rho_c[0] * e_chg_0[n, 0] - (1.0 / rho_d[0]) * e_dis_0[n, 0])
                        m.addConstr(now_next_1[n, 0] == rho[1] * z[1] + rho_c[1] * e_chg_1[n, 0] - (1.0 / rho_d[1]) * e_dis_1[n, 0])
                        m.addConstr(now_next_2[n, 0] == rho[2] * z[2] + rho_c[2] * e_chg_2[n, 0] - (1.0 / rho_d[2]) * e_dis_2[n, 0])
                    else:
                        m.addConstr(now_next_0[n, h] == rho[0] * now_next_0[n, h - 1] + rho_c[0] * e_chg_0[n, h] - (1.0 / rho_d[0]) * e_dis_0[n, h])
                        m.addConstr(now_next_1[n, h] == rho[1] * now_next_1[n, h - 1] + rho_c[1] * e_chg_1[n, h] - (1.0 / rho_d[1]) * e_dis_1[n, h])
                        m.addConstr(now_next_2[n, h] == rho[2] * now_next_2[n, h - 1] + rho_c[2] * e_chg_2[n, h] - (1.0 / rho_d[2]) * e_dis_2[n, h])
                m.addConstr(now[n, 0] == now_next_0[n, H - 1])
                m.addConstr(now[n, 1] == now_next_1[n, H - 1])
                m.addConstr(now[n, 2] == now_next_2[n, H - 1])

            if t < T - 1:
                for n in range(N):
                    sqrtq = math.sqrt(float(q[n]))
                    m.addConstr(
                        sqrtq * gamma >= sqrtq * alpha[n] - (mu[n] - lmbda[n]),
                        name=f'gamma_const[{n}]'
                    )
                    m.addConstr(mu[n] + lmbda[n] == beta / math.sqrt(N) + phi[n], name=f'robust_norm[{n}]')

            m.update()
            stage_models.append(m)
        models.append(stage_models)
    return models

def make_uniform_transition_matrix():
    return [np.ones((1, N), dtype=float) / N] + [
        np.ones((N, N), dtype=float) / N
        for _ in range(T)
    ]

def set_transition_weights_for_models(models, transition_matrix):
    for t, stage_models in enumerate(models):
        for idx, model in enumerate(stage_models):
            q = normalize_probability_rows(np.asarray(transition_matrix[t + 1][idx]).reshape(1, -1))[0]

            price_vars = [model.getVarByName(f'x[{h}]') for h in range(H)]
            for n in range(N):
                for h in range(H):
                    unmet = model.getVarByName(f'unmet[{n},{h}]')
                    if unmet is not None:
                        price_coeff = -float(price_vars[h].Obj) if price_vars[h] is not None else 0.0
                        unmet.Obj = float(2.0 * q[n] * price_coeff)

            if t < T - 1:
                for n in range(N):
                    qn = float(q[n])
                    sqrt_q = math.sqrt(qn)
                    gamma = model.getVarByName('gamma')
                    alpha = model.getVarByName(f'alpha[{n}]')
                    mu = model.getVarByName(f'mu[{n}]')
                    lmbda = model.getVarByName(f'lambda[{n}]')
                    gamma_const = model.getConstrByName(f'gamma_const[{n}]')
                    if mu is not None:
                        mu.Obj = sqrt_q
                    if lmbda is not None:
                        lmbda.Obj = -sqrt_q
                    if (
                        gamma_const is not None and gamma is not None and alpha is not None
                        and mu is not None and lmbda is not None
                    ):
                        model.chgCoeff(gamma_const, gamma, sqrt_q)
                        model.chgCoeff(gamma_const, alpha, -sqrt_q)
                        model.chgCoeff(gamma_const, mu, 1.0)
                        model.chgCoeff(gamma_const, lmbda, -1.0)
            model.update()

def build_quarter_model_templates(weekly_power, weekly_price):
    daily_power_all = split_daily(weekly_power, T)
    daily_price_all = split_daily(weekly_price, T)
    daily_power = daily_power_all[:, :N, :]
    daily_price = daily_price_all[:, :N, :]
    price_scale = float(np.percentile(daily_price, 90))
    if not np.isfinite(price_scale) or price_scale <= 0:
        price_scale = 1.0
    daily_price_run = daily_price / price_scale
    return build_stage_models(
        daily_power,
        daily_price_run,
        make_uniform_transition_matrix(),
        eps=0.0,
    )

def set_eps_objective_for_models(models, eps):
    sqrt_eps = math.sqrt(max(float(eps), 0.0))
    for t, stage_models in enumerate(models):
        if t >= T - 1:
            continue
        for model in stage_models:
            beta = model.getVarByName('beta')
            if beta is not None:
                beta.Obj = sqrt_eps
            for n in range(N):
                phi = model.getVarByName(f'phi[{n}]')
                if phi is not None:
                    phi.Obj = sqrt_eps
            model.update()

def copy_stage_model_templates(model_templates, eps, transition_matrix):
    models = []
    for stage_templates in model_templates:
        stage_models = []
        for template in stage_templates:
            template.update()
            model = template.copy()
            model.Params.OutputFlag = OutputFlag
            model.Params.LogToConsole = 0
            model.Params.DualReductions = 0
            model.Params.Threads = 1
            model.Params.Seed = BASE_SEED
            stage_models.append(model)
        models.append(stage_models)
    set_transition_weights_for_models(models, transition_matrix)
    set_eps_objective_for_models(models, eps)
    return models

In [ ]:
# SDDP Solver and OOS Evaluation
def make_forward_uniforms(base_seed, quarter, num_iter, T_):
    rng = np.random.default_rng(base_seed + 1000 * int(quarter))
    return rng.random((int(num_iter) + 1, int(T_) + 1))

def create_sample_path_from_uniforms(n, T_, transition_matrix, uniforms):
    path = []
    uniforms = np.asarray(uniforms, dtype=float)
    if uniforms.shape[0] < T_ + 1:
        raise ValueError('uniforms must have at least T_ + 1 entries.')
    for t in range(T_ + 1):
        if t == 0:
            p = np.asarray(transition_matrix[0][0], dtype=float)
        else:
            p = np.asarray(transition_matrix[t][path[-1]], dtype=float)
        p = normalize_probability_rows(p.reshape(1, -1))[0]
        cdf = np.cumsum(p)
        cdf[-1] = 1.0
        u = float(uniforms[t])
        idx = int(np.searchsorted(cdf, u, side="right"))
        idx = min(max(idx, 0), n - 1)
        path.append(idx)
    return path

def actual_storage_and_revenue(power_vec, price_vec, x_vec, prev_storage):
    power_vec = np.asarray(power_vec, dtype=float)
    price_vec = np.asarray(price_vec, dtype=float)
    x_vec = np.asarray(x_vec, dtype=float)
    curr_storage = np.asarray(prev_storage, dtype=float).copy()
    revenue = 0.0
    for h in range(H):
        revenue += x_vec[h] * price_vec[h]
        curr_storage = rho * curr_storage
        if x_vec[h] < power_vec[h]:
            curr_excess = power_vec[h] - x_vec[h]
            if curr_storage[2] + rho_c[2] * curr_excess >= cap[2]:
                curr_excess = curr_excess - (1.0 / rho_c[2]) * (cap[2] - curr_storage[2])
                curr_storage[2] = cap[2]
                if curr_storage[1] + rho_c[1] * curr_excess >= cap[1]:
                    curr_excess = curr_excess - (1.0 / rho_c[1]) * (cap[1] - curr_storage[1])
                    curr_storage[1] = cap[1]
                    if curr_storage[0] + rho_c[0] * curr_excess >= cap[0]:
                        curr_excess = curr_excess - (1.0 / rho_c[0]) * (cap[0] - curr_storage[0])
                        curr_storage[0] = cap[0]
                    else:
                        curr_storage[0] = curr_storage[0] + rho_c[0] * curr_excess
                else:
                    curr_storage[1] = curr_storage[1] + rho_c[1] * curr_excess
            else:
                curr_storage[2] = curr_storage[2] + rho_c[2] * curr_excess
        else:
            curr_short = x_vec[h] - power_vec[h]
            if curr_short <= curr_storage[2] * rho_d[2]:
                curr_storage[2] = curr_storage[2] - (1.0 / rho_d[2]) * curr_short
            else:
                curr_short = curr_short - curr_storage[2] * rho_d[2]
                curr_storage[2] = 0.0
                if curr_short <= curr_storage[1] * rho_d[1]:
                    curr_storage[1] = curr_storage[1] - (1.0 / rho_d[1]) * curr_short
                else:
                    curr_short = curr_short - curr_storage[1] * rho_d[1]
                    curr_storage[1] = 0.0
                    if curr_short <= curr_storage[0] * rho_d[0]:
                        curr_storage[0] = curr_storage[0] - (1.0 / rho_d[0]) * curr_short
                    else:
                        curr_short = curr_short - curr_storage[0] * rho_d[0]
                        curr_storage[0] = 0.0
                        revenue = revenue - 2.0 * price_vec[h] * curr_short
    return curr_storage, float(revenue)

def solve_policy(Markov_states_power, Markov_states_price, transition_matrix, eps,
                 num_iter=None, initial_cuts=None, cut_records=None, forward_uniforms=None,
                 model_templates=None):
    if model_templates is None:
        models = build_stage_models(Markov_states_power, Markov_states_price, transition_matrix, eps)
    else:
        models = copy_stage_model_templates(model_templates, eps, transition_matrix)
    if initial_cuts is not None:
        add_neural_initial_cuts_to_models(models, initial_cuts, transition_matrix)

    run_iter = NUM_ITER if num_iter is None else int(num_iter)
    if forward_uniforms is None:
        rng = np.random.default_rng(BASE_SEED)
        forward_uniforms = rng.random((run_iter + 1, T + 1))
    forward_uniforms = np.asarray(forward_uniforms, dtype=float)
    if forward_uniforms.shape[0] < run_iter + 1:
        raise ValueError('forward_uniforms must have at least run_iter + 1 rows.')
    if forward_uniforms.shape[1] < T + 1:
        raise ValueError('forward_uniforms must have at least T + 1 columns.')

    path = create_sample_path_from_uniforms(N, T, transition_matrix, forward_uniforms[0])
    x_bar = [INIT_LEVEL.copy()]
    decisions = []
    obj_trace = []

    def forward_pass(sample_path):
        storage_path = []
        decision_path = []
        prev_storage = INIT_LEVEL.copy()
        neg_profit = 0.0
        for t in range(T):
            idx = sample_path[t]
            model = models[t][idx]
            set_storage_rhs(model, prev_storage)
            optimize_or_raise(model, f'forward t={t} idx={idx}')
            x_vec = np.asarray([model.getVarByName(f'x[{h}]').X for h in range(H)], dtype=float)
            next_idx = sample_path[t + 1]
            prev_storage = np.asarray([model.getVarByName(f'now[{next_idx},{l}]').X for l in range(K)], dtype=float)
            revenue = 0.0
            for h in range(H):
                unmet = model.getVarByName(f'unmet[{next_idx},{h}]').X
                revenue += float(Markov_states_price[t][idx][h]) * x_vec[h] - 2.0 * float(Markov_states_price[t][idx][h]) * unmet
            storage_path.append(prev_storage.copy())
            decision_path.append(x_vec)
            neg_profit -= revenue
        return storage_path, decision_path, neg_profit


    x_bar, decisions, first_neg_profit = forward_pass(path)
    obj_trace.append(float(first_neg_profit))

    for it in range(run_iter):
        obj_next = np.zeros(N, dtype=float)
        dual_next = np.zeros((N, K), dtype=float)

        for idx in range(N):
            model = models[T - 1][idx]
            set_storage_rhs(model, x_bar[T - 2] if T >= 2 else INIT_LEVEL)
            optimize_or_raise(model, f'backward terminal it={it} idx={idx}')
            obj_next[idx] = model.ObjVal
            for l in range(K):
                dual_next[idx, l] = model.getConstrByName(f'grad_constr_{l}').Pi

        for t in range(T - 2, -1, -1):
            add_alpha_cuts_to_stage(
                models, t, it, obj_next, dual_next, x_bar[t],
                transition_matrix=transition_matrix,
                cut_records=cut_records,
            )
            obj_cur = np.zeros(N, dtype=float)
            dual_cur = np.zeros((N, K), dtype=float)
            prev_storage = INIT_LEVEL if t == 0 else x_bar[t - 1]
            for idx in range(N):
                model = models[t][idx]
                set_storage_rhs(model, prev_storage)
                optimize_or_raise(model, f'backward it={it} t={t} idx={idx}')
                obj_cur[idx] = model.ObjVal
                for l in range(K):
                    dual_cur[idx, l] = model.getConstrByName(f'grad_constr_{l}').Pi
            obj_next, dual_next = obj_cur, dual_cur

        path = create_sample_path_from_uniforms(N, T, transition_matrix, forward_uniforms[it + 1])
        x_bar, decisions, neg_profit = forward_pass(path)
        obj_trace.append(float(neg_profit))

        should_print = VERBOSE and ((it + 1) % max(1, run_iter // 10) == 0 or it == 0)
        if should_print:
            print(
                f'iter={it + 1:03d}/{run_iter:03d} '
                f'forward_negative_profit={neg_profit / 1e5:.4f}'
            )

    return {
        'models': models,
        'obj_trace': obj_trace,
        'iterations_run': int(run_iter),
    }

def evaluate_oos(policy, Markov_states_power_train, Markov_states_price_train, daily_wind_oos, daily_power_oos,
                 daily_price_oos_scaled, daily_price_oos_raw, transition_info, mode, eps,
                 transition_row_cache=None):
    if daily_power_oos is None or daily_power_oos.shape[1] == 0:
        return None
    if VERBOSE:
        print('OOS_Start')
    models = policy['models']
    n_oos = daily_power_oos.shape[1]

    if transition_row_cache is None:
        transition_row_cache = build_oos_transition_cache(
            mode,
            daily_wind_oos,
            daily_power_oos,
            daily_price_oos_scaled,
            transition_info,
        )
    transition_row_cache = np.asarray(transition_row_cache, dtype=float)
    if transition_row_cache.shape != (T, n_oos, N):
        raise ValueError(
            f'transition_row_cache shape mismatch: got {transition_row_cache.shape}, '
            f'expected {(T, n_oos, N)}.'
        )

    eval_models = []
    try:
        for t in range(T):
            model = models[t][0].copy()
            model.Params.OutputFlag = OutputFlag
            model.Params.LogToConsole = 0
            model.Params.DualReductions = 0
            model.Params.Threads = 1
            model.Params.Seed = BASE_SEED
            eval_models.append(model)

        negative_profit_raw = []
        for kk in range(n_oos):
            storage = INIT_LEVEL.copy()
            neg_raw = 0.0
            for t in range(T):
                model = eval_models[t]
                weights = transition_row_cache[t, kk]
                update_oos_objective_and_weights(model, daily_price_oos_scaled[t, kk], weights, eps)
                set_storage_rhs(model, storage)
                optimize_or_raise(model, f'OOS kk={kk} t={t}')
                x_vec = np.asarray([model.getVarByName(f'x[{h}]').X for h in range(H)], dtype=float)
                storage, revenue_raw = actual_storage_and_revenue(
                    daily_power_oos[t + 1, kk],
                    daily_price_oos_raw[t, kk],
                    x_vec,
                    storage,
                )
                neg_raw -= revenue_raw
            negative_profit_raw.append(float(neg_raw / 1e5))
            if VERBOSE:
                print(f'OOS[{kk + 1}/{n_oos}] negative_profit_raw={negative_profit_raw[-1]:.4f}')
        return np.asarray(negative_profit_raw, dtype=float)
    finally:
        for model in eval_models:
            model.dispose()

def dispose_policy(policy):
    if not policy:
        return
    for stage_models in policy.get('models', []):
        for model in stage_models:
            try:
                model.dispose()
            except Exception:
                pass

In [ ]:
# NN functions
def neural_context_features(stage_t, daily_power, daily_price):
    power_now = np.asarray(daily_power[stage_t], dtype=float)
    price_now = np.asarray(daily_price[stage_t], dtype=float)
    next_t = min(stage_t + 1, daily_power.shape[0] - 1)
    power_next = np.asarray(daily_power[next_t], dtype=float)
    price_next = np.asarray(daily_price[next_t], dtype=float)
    return np.asarray([
        float(stage_t) / max(T - 1, 1),
        float(np.mean(power_now)),
        float(np.std(power_now)),
        float(np.mean(price_now)),
        float(np.std(price_now)),
        float(np.mean(power_next)),
        float(np.std(power_next)),
        float(np.mean(price_next)),
        float(np.std(price_next)),
    ], dtype=float)

def collect_sddp_cuts(Markov_states_power, Markov_states_price, transition_matrix, eps,
                      teacher_iter=None, forward_uniforms=None, model_templates=None):
    cut_records = []
    policy = solve_policy(
        Markov_states_power,
        Markov_states_price,
        transition_matrix,
        eps,
        num_iter=NUM_ITER if teacher_iter is None else int(teacher_iter),
        cut_records=cut_records,
        forward_uniforms=forward_uniforms,
        model_templates=model_templates,
    )
    return cut_records, policy

def build_neural_training_dataset(cut_records, daily_power, daily_price):
    if not cut_records:
        raise ValueError('No teacher SDDP cuts were collected for neural warm start.')
    x_rows = []
    y_rows = []
    for t in range(T - 1):
        stage_records = [r for r in cut_records if int(r['stage']) == t]
        stage_records.sort(
            key=lambda r: (
                int(r.get('iteration', 0)),
                int(r.get('node', -1)),
                int(r.get('child', -1)),
            )
        )
        if len(stage_records) >= NUM_NEURAL_CUTS:
            pick = np.linspace(0, len(stage_records) - 1, NUM_NEURAL_CUTS).round().astype(int)
            chosen = [stage_records[int(i)] for i in pick]
        elif stage_records:
            chosen = list(stage_records)
            while len(chosen) < NUM_NEURAL_CUTS:
                chosen.append(stage_records[-1])
        else:
            chosen = [{'a': 0.0, 'b': [0.0] * K} for _ in range(NUM_NEURAL_CUTS)]
        coeffs = []
        for rec in chosen[:NUM_NEURAL_CUTS]:
            slope = np.asarray(rec['b'], dtype=float)
            if slope.shape[0] != K:
                raise ValueError(f'Expected neural cut slope dimension {K}, got {slope.shape[0]}.')
            coeffs.extend([float(rec['a']), *slope.tolist()])
        x_rows.append(neural_context_features(t, daily_power, daily_price))
        y_rows.append(np.asarray(coeffs, dtype=float))
    return np.vstack(x_rows), np.vstack(y_rows)

def sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def train_cut_predictor(X, Y):
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)
    x_mean = X.mean(axis=0, keepdims=True)
    x_std = X.std(axis=0, keepdims=True)
    x_std[x_std < 1e-6] = 1.0
    y_mean = Y.mean(axis=0, keepdims=True)
    y_std = Y.std(axis=0, keepdims=True)
    y_std[y_std < 1e-6] = 1.0
    X_t = torch.tensor((X - x_mean) / x_std, dtype=torch.float32, device=DEVICE)
    Y_t = torch.tensor((Y - y_mean) / y_std, dtype=torch.float32, device=DEVICE)

    model = nn.Sequential(
        nn.Linear(X.shape[1], NEURAL_HIDDEN),
        nn.ReLU(),
        nn.Linear(NEURAL_HIDDEN, NEURAL_HIDDEN),
        nn.ReLU(),
        nn.Linear(NEURAL_HIDDEN, Y.shape[1]),
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=NEURAL_LR)
    loss_fn = nn.MSELoss()
    model.train()
    for epoch in range(int(NEURAL_EPOCHS)):
        opt.zero_grad()
        loss = loss_fn(model(X_t), Y_t)
        loss.backward()
        opt.step()
        if VERBOSE and (epoch == 0 or (epoch + 1) % max(1, int(NEURAL_EPOCHS) // 5) == 0):
            print(f'neural_cut_predictor epoch={epoch + 1} loss={float(loss.item()):.6g}')
    return {'model': model, 'x_mean': x_mean, 'x_std': x_std, 'y_mean': y_mean, 'y_std': y_std}

def predict_initial_cuts(predictor, daily_power, daily_price):
    model = predictor['model'].to(DEVICE)
    model.eval()
    contexts = np.vstack([neural_context_features(t, daily_power, daily_price) for t in range(T - 1)]).astype(np.float32)
    Xn = (contexts - predictor['x_mean']) / predictor['x_std']
    x_tensor = torch.tensor(Xn, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        pred = model(x_tensor).cpu().numpy()
    pred = pred * predictor['y_std'] + predictor['y_mean']

    initial_cuts = {}
    for row_idx, t in enumerate(range(T - 1)):
        coeffs = pred[row_idx].reshape(NUM_NEURAL_CUTS, K + 1)
        stage_cuts = []
        for k_idx in range(NUM_NEURAL_CUTS):
            a = float(coeffs[k_idx, 0])
            b = np.asarray(coeffs[k_idx, 1:], dtype=float)
            if np.isfinite(a) and np.all(np.isfinite(b)):
                stage_cuts.append({'a': a, 'b': b})
        initial_cuts[t] = stage_cuts
    return initial_cuts

def add_neural_initial_cuts_to_models(models, initial_cuts, transition_matrix):
    for t, stage_cuts in initial_cuts.items():
        if t >= T - 1:
            continue
        for idx, model in enumerate(models[t]):
            for cut_idx, cut in enumerate(stage_cuts):
                slope = np.asarray(cut['b'], dtype=float)
                intercept = float(cut['a'])
                if slope.shape[0] != K:
                    raise ValueError(f'Expected neural cut slope dimension {K}, got {slope.shape[0]}.')

                for n in range(N):
                    alpha_var = model.getVarByName(f'alpha[{n}]')
                    now_vars = [model.getVarByName(f'now[{n},{l}]') for l in range(K)]
                    if alpha_var is None or any(v is None for v in now_vars):
                        continue

                    expr = alpha_var - gurobipy.quicksum(
                        float(slope[l]) * now_vars[l] for l in range(K)
                    )
                    model.addConstr(
                        expr >= intercept,
                        name=f'neural_multicut_init_t{t}_k{cut_idx}_node{idx}_child{n}',
                    )
            model.update()

In [ ]:
# Experiment loop
def normalize_mode_name(method):
    return str(method).strip().replace('-', '_').upper()

def run_quarter(method, q, weekly_wind, weekly_power, weekly_price, forward_uniforms,
                base_model_templates=None, neural_teacher_cache=None):
    runtime_start = time.perf_counter()
    method_u = normalize_mode_name(method)
    if VERBOSE:
        print('\n' + '=' * 80)
        print(f'Method={method_u} | Quarter={q} | N={N} | NUM_DAYS={NUM_DAYS} | T={T}')

    time_transition_seconds = 0.0
    time_eps_tuning_seconds = 0.0
    time_sddp_solve_seconds = 0.0
    time_validation_eval_seconds = 0.0
    time_test_eval_seconds = 0.0
    time_nn_teacher_sddp_seconds = None
    time_nn_training_seconds = None
    time_nn_prediction_seconds = None
    time_nn_final_forward_seconds = None
    time_nn_total_with_teacher_seconds = None
    time_nn_inference_excluding_teacher_seconds = None
    nn_teacher_reused = False
    internal_cv_mean = None
    internal_cv_train_n = None
    internal_cv_val_size = None

    daily_wind_all = split_daily(weekly_wind, T) if weekly_wind is not None else None
    daily_power_all = split_daily(weekly_power, T)
    daily_price_all = split_daily(weekly_price, T)

    holdout_count = daily_power_all.shape[1] - N
    if VAL_SIZE <= 0:
        raise ValueError(f'VAL_SIZE must be positive; got {VAL_SIZE}.')
    if VAL_SIZE > holdout_count:
        raise ValueError(
            f'VAL_SIZE={VAL_SIZE} exceeds available holdout trajectories after N={N}: {holdout_count}.'
        )
    if VAL_SIZE == holdout_count:
        raise ValueError(
            f'VAL_SIZE={VAL_SIZE} leaves no separate test trajectories; choose VAL_SIZE < {holdout_count}.'
        )

    daily_wind = daily_wind_all[:, :N, :] if daily_wind_all is not None else None
    daily_power = daily_power_all[:, :N, :]
    daily_price = daily_price_all[:, :N, :]

    daily_wind_val = daily_wind_all[:, N:N + VAL_SIZE, :] if daily_wind_all is not None else None
    daily_power_val = daily_power_all[:, N:N + VAL_SIZE, :]
    daily_price_val = daily_price_all[:, N:N + VAL_SIZE, :]

    daily_wind_test = daily_wind_all[:, N + VAL_SIZE:, :] if daily_wind_all is not None else None
    daily_power_test = daily_power_all[:, N + VAL_SIZE:, :]
    daily_price_test = daily_price_all[:, N + VAL_SIZE:, :]

    price_scale = float(np.percentile(daily_price, 90))
    if not np.isfinite(price_scale) or price_scale <= 0:
        price_scale = 1.0
    daily_price_run = daily_price / price_scale
    daily_price_val_run = daily_price_val / price_scale
    daily_price_test_run = daily_price_test / price_scale

    selected_gmm_K = None
    selected_gmm_reg_covar = None
    selected_nw_bandwidth = None
    neural_variant = None

    t0 = time.perf_counter()
    if method_u == 'IND':
        transition_matrix = [np.ones((1, N), dtype=float) / N] + [np.ones((N, N), dtype=float) / N for _ in range(T)]
        transition_info = {'feature_name': 'POWER_PRICE'}
        method_label = 'IND'
        transition_eval_mode = 'IND'
        if VERBOSE:
            transition_diagnostics('IND', transition_matrix)
    elif method_u == 'NW':
        transition_matrix = None
        transition_info = {'feature_name': 'POWER_PRICE'}
        method_label = 'NW-DRO'
        transition_eval_mode = 'NW'
        if VERBOSE:
            print('NW transition will be built after internal bandwidth selection.')
    elif method_u == 'GMM':
        gmm_raw = select_feature_raw('POWER_PRICE', daily_wind, daily_power, daily_price_run)
        transition_matrix, transition_info = build_gmm_transition_matrix(
            gmm_raw,
            feature_name='POWER_PRICE',
        )
        selected_gmm_K = transition_info['selected_gmm_K']
        selected_gmm_reg_covar = transition_info.get('selected_gmm_reg_covar')
        method_label = 'GMM-DRO'
        transition_eval_mode = 'GMM'
        if VERBOSE:
            transition_diagnostics('GMM', transition_matrix)
    elif method_u == 'NN':
        if not USE_NEURAL_WARM_START:
            raise ValueError('USE_NEURAL_WARM_START must be True for NN.')
        transition_matrix = [np.ones((1, N), dtype=float) / N] + [np.ones((N, N), dtype=float) / N for _ in range(T)]
        transition_info = {'feature_name': 'POWER_PRICE'}
        method_label = 'N-SDDP'
        transition_eval_mode = 'IND'
        neural_variant = 'fast'
        if VERBOSE:
            transition_diagnostics('NN', transition_matrix)
    else:
        raise ValueError('Method must be IND, NW, GMM, or NN.')
    time_transition_seconds = float(time.perf_counter() - t0)

    if method_u in {'NW', 'GMM'}:
        val_transition_cache = None
        test_transition_cache = None
    else:
        t0 = time.perf_counter()
        val_transition_cache = build_oos_transition_cache(
            transition_eval_mode,
            daily_wind_val,
            daily_power_val,
            daily_price_val_run,
            transition_info,
        )
        test_transition_cache = build_oos_transition_cache(
            transition_eval_mode,
            daily_wind_test,
            daily_power_test,
            daily_price_test_run,
            transition_info,
        )
        time_transition_seconds += float(time.perf_counter() - t0)

    if method_u == 'NN':
        best_eps = 0.0
        teacher_policy = None
        best_policy = None
        try:
            torch.manual_seed(BASE_SEED + 50000 + 1000 * int(q))
            np.random.seed(BASE_SEED + 50000 + 1000 * int(q))
            cached_teacher = None
            if neural_teacher_cache is not None:
                cached_teacher = neural_teacher_cache.get(int(q))

            if cached_teacher is not None and cached_teacher.get('cut_records'):
                teacher_cuts = list(cached_teacher['cut_records'])
                time_nn_teacher_sddp_seconds = 0.0
                nn_teacher_reused = True
                if VERBOSE:
                    print(
                        f'Method=N-SDDP | Quarter={q} reusing '
                        f"{len(teacher_cuts)} IND teacher cuts"
                    )
            else:
                teacher_forward_uniforms = make_forward_uniforms(BASE_SEED + 70000, q, NEURAL_TEACHER_ITER, T)

                t0 = time.perf_counter()
                teacher_cuts, teacher_policy = collect_sddp_cuts(
                    daily_power,
                    daily_price_run,
                    transition_matrix,
                    eps=0.0,
                    teacher_iter=NEURAL_TEACHER_ITER,
                    forward_uniforms=teacher_forward_uniforms,
                    model_templates=base_model_templates,
                )
                time_nn_teacher_sddp_seconds = float(time.perf_counter() - t0)
                if neural_teacher_cache is not None:
                    neural_teacher_cache[int(q)] = {
                        'cut_records': list(teacher_cuts),
                        'source': 'NN_teacher_fallback',
                        'teacher_iter': int(NEURAL_TEACHER_ITER),
                    }

            X_train, Y_train = build_neural_training_dataset(teacher_cuts, daily_power, daily_price_run)

            sync_if_cuda()
            t0 = time.perf_counter()
            predictor = train_cut_predictor(X_train, Y_train)
            sync_if_cuda()
            time_nn_training_seconds = float(time.perf_counter() - t0)

            sync_if_cuda()
            t0 = time.perf_counter()
            initial_cuts = predict_initial_cuts(predictor, daily_power, daily_price_run)
            sync_if_cuda()
            time_nn_prediction_seconds = float(time.perf_counter() - t0)

            dispose_policy(teacher_policy)
            teacher_policy = None

            t0 = time.perf_counter()
            best_policy = solve_policy(
                daily_power,
                daily_price_run,
                transition_matrix,
                eps=0.0,
                num_iter=0,
                initial_cuts=initial_cuts,
                forward_uniforms=forward_uniforms,
                model_templates=base_model_templates,
            )
            time_nn_final_forward_seconds = float(time.perf_counter() - t0)
            time_sddp_solve_seconds = float(time_nn_final_forward_seconds)

            t0 = time.perf_counter()
            val_neg_raw = evaluate_oos(
                best_policy,
                daily_power,
                daily_price_run,
                daily_wind_val,
                daily_power_val,
                daily_price_val_run,
                daily_price_val,
                transition_info,
                transition_eval_mode,
                best_eps,
                transition_row_cache=val_transition_cache,
            )
            time_validation_eval_seconds = float(time.perf_counter() - t0)
            if val_neg_raw is None:
                raise ValueError('Validation split is empty; choose a positive VAL_SIZE with available holdout trajectories.')
            VAL_mean = float(np.mean(val_neg_raw))

            t0 = time.perf_counter()
            test_neg_raw = evaluate_oos(
                best_policy,
                daily_power,
                daily_price_run,
                daily_wind_test,
                daily_power_test,
                daily_price_test_run,
                daily_price_test,
                transition_info,
                transition_eval_mode,
                best_eps,
                transition_row_cache=test_transition_cache,
            )
            time_test_eval_seconds = float(time.perf_counter() - t0)
            if test_neg_raw is None:
                raise ValueError('Test split is empty after validation split; choose a smaller VAL_SIZE.')
        finally:
            if teacher_policy is not None:
                dispose_policy(teacher_policy)

        time_nn_total_with_teacher_seconds = float(
            time_nn_teacher_sddp_seconds
            + time_nn_training_seconds
            + time_nn_prediction_seconds
            + time_nn_final_forward_seconds
            + time_validation_eval_seconds
            + time_test_eval_seconds
        )
        time_nn_inference_excluding_teacher_seconds = float(
            time_nn_prediction_seconds + time_nn_final_forward_seconds + time_test_eval_seconds
        )
    elif method_u == 'IND':
        best_eps = 0.0

        ind_cut_records = [] if neural_teacher_cache is not None else None
        t0 = time.perf_counter()
        best_policy = solve_policy(
            daily_power,
            daily_price_run,
            transition_matrix,
            best_eps,
            forward_uniforms=forward_uniforms,
            model_templates=base_model_templates,
            cut_records=ind_cut_records,
        )
        time_sddp_solve_seconds = float(time.perf_counter() - t0)
        if neural_teacher_cache is not None and ind_cut_records:
            neural_teacher_cache[int(q)] = {
                'cut_records': list(ind_cut_records),
                'source': 'IND',
                'teacher_iter': int(NUM_ITER),
            }
            if VERBOSE:
                print(
                    f'Method=IND | Quarter={q} cached '
                    f"{len(ind_cut_records)} teacher cuts for NN"
                )

        t0 = time.perf_counter()
        val_neg_raw = evaluate_oos(
            best_policy,
            daily_power,
            daily_price_run,
            daily_wind_val,
            daily_power_val,
            daily_price_val_run,
            daily_price_val,
            transition_info,
            transition_eval_mode,
            best_eps,
            transition_row_cache=val_transition_cache,
        )
        time_validation_eval_seconds = float(time.perf_counter() - t0)
        if val_neg_raw is None:
            dispose_policy(best_policy)
            raise ValueError('Validation split is empty; choose a positive VAL_SIZE with available holdout trajectories.')
        VAL_mean = float(np.mean(val_neg_raw))

        t0 = time.perf_counter()
        test_neg_raw = evaluate_oos(
            best_policy,
            daily_power,
            daily_price_run,
            daily_wind_test,
            daily_power_test,
            daily_price_test_run,
            daily_price_test,
            transition_info,
            transition_eval_mode,
            best_eps,
            transition_row_cache=test_transition_cache,
        )
        time_test_eval_seconds = float(time.perf_counter() - t0)
        if test_neg_raw is None:
            dispose_policy(best_policy)
            raise ValueError('Test split is empty after validation split; choose a smaller VAL_SIZE.')
    elif method_u in {'NW', 'GMM'}:
        best = None
        eps_candidates = list(EPS_LIST)
        bandwidth_candidates = get_nw_bandwidth_candidates() if method_u == 'NW' else [None]
        cv_num_iter = int(min(int(EPS_CV_MAX_ITER), int(MAX_ITER)))

        full_train_n = int(N)
        internal_cv_train_n = int(IN_SAMPLE_CV_TRAIN_N)
        internal_cv_val_size = int(VAL_SIZE)
        if internal_cv_train_n <= 0 or internal_cv_val_size <= 0:
            raise ValueError(
                f'Internal CV split must be positive; got train={internal_cv_train_n}, '
                f'val={internal_cv_val_size}.'
            )
        if internal_cv_train_n + internal_cv_val_size > full_train_n:
            raise ValueError(
                f'Internal CV split train+val={internal_cv_train_n + internal_cv_val_size} '
                f'exceeds final in-sample N={full_train_n}.'
            )

        daily_wind_cv_train = (
            daily_wind[:, :internal_cv_train_n, :]
            if daily_wind is not None else None
        )
        daily_power_cv_train = daily_power[:, :internal_cv_train_n, :]
        daily_price_cv_train = daily_price[:, :internal_cv_train_n, :]
        daily_wind_cv_val = (
            daily_wind[:, internal_cv_train_n:internal_cv_train_n + internal_cv_val_size, :]
            if daily_wind is not None else None
        )
        daily_power_cv_val = daily_power[:, internal_cv_train_n:internal_cv_train_n + internal_cv_val_size, :]
        daily_price_cv_val = daily_price[:, internal_cv_train_n:internal_cv_train_n + internal_cv_val_size, :]

        cv_price_scale = float(np.percentile(daily_price_cv_train, 90))
        if not np.isfinite(cv_price_scale) or cv_price_scale <= 0:
            cv_price_scale = 1.0
        daily_price_cv_train_run = daily_price_cv_train / cv_price_scale
        daily_price_cv_val_run = daily_price_cv_val / cv_price_scale

        if VERBOSE:
            print(
                f'Method={method_label} | Quarter={q} internal CV within N={full_train_n}: '
                f'train={internal_cv_train_n}, val={internal_cv_val_size}, '
                f'CV_iter={cv_num_iter}'
            )

        t_eps_start = time.perf_counter()
        old_global_N = int(globals()['N'])
        cv_model_templates = None
        try:
            globals()['N'] = internal_cv_train_n

            t0 = time.perf_counter()
            cv_model_templates = build_stage_models(
                daily_power_cv_train,
                daily_price_cv_train_run,
                make_uniform_transition_matrix(),
                eps=0.0,
            )
            time_sddp_solve_seconds += float(time.perf_counter() - t0)

            for nw_bandwidth in bandwidth_candidates:
                if method_u == 'NW':
                    t0 = time.perf_counter()
                    cand_transition_matrix, cand_transition_info = build_power_price_nw_transition_matrix(
                        daily_power_cv_train,
                        daily_price_cv_train_run,
                        bandwidth=nw_bandwidth,
                    )
                    time_transition_seconds += float(time.perf_counter() - t0)
                    cand_transition_eval_mode = 'NW'

                    if VERBOSE:
                        print(
                            f'Method={method_label} | Quarter={q} | '
                            f'internal CV NW bandwidth={nw_bandwidth:g}'
                        )
                        transition_diagnostics(
                            f'NW internal bandwidth={nw_bandwidth:g}',
                            cand_transition_matrix,
                        )
                else:
                    t0 = time.perf_counter()
                    cv_gmm_raw = select_feature_raw(
                        'POWER_PRICE',
                        daily_wind_cv_train,
                        daily_power_cv_train,
                        daily_price_cv_train_run,
                    )
                    cand_transition_matrix, cand_transition_info = build_gmm_transition_matrix(
                        cv_gmm_raw,
                        feature_name='POWER_PRICE',
                    )
                    time_transition_seconds += float(time.perf_counter() - t0)
                    cand_transition_eval_mode = 'GMM'

                    if VERBOSE:
                        transition_diagnostics('GMM internal CV', cand_transition_matrix)

                t0 = time.perf_counter()
                cand_cv_val_transition_cache = build_oos_transition_cache(
                    cand_transition_eval_mode,
                    daily_wind_cv_val,
                    daily_power_cv_val,
                    daily_price_cv_val_run,
                    cand_transition_info,
                )
                time_transition_seconds += float(time.perf_counter() - t0)

                for eps_candidate in eps_candidates:
                    eps_value = float(eps_candidate)

                    t0 = time.perf_counter()
                    policy = solve_policy(
                        daily_power_cv_train,
                        daily_price_cv_train_run,
                        cand_transition_matrix,
                        eps_value,
                        num_iter=cv_num_iter,
                        forward_uniforms=forward_uniforms,
                        model_templates=cv_model_templates,
                    )
                    time_sddp_solve_seconds += float(time.perf_counter() - t0)

                    t0 = time.perf_counter()
                    cv_neg_raw = evaluate_oos(
                        policy,
                        daily_power_cv_train,
                        daily_price_cv_train_run,
                        daily_wind_cv_val,
                        daily_power_cv_val,
                        daily_price_cv_val_run,
                        daily_price_cv_val,
                        cand_transition_info,
                        cand_transition_eval_mode,
                        eps_value,
                        transition_row_cache=cand_cv_val_transition_cache,
                    )
                    time_validation_eval_seconds += float(time.perf_counter() - t0)
                    dispose_policy(policy)

                    if cv_neg_raw is None:
                        raise ValueError(
                            'Internal validation split is empty; choose a positive '
                            'VAL_SIZE within N.'
                        )
                    cand_internal_cv_mean = float(np.mean(cv_neg_raw))

                    if VERBOSE:
                        if method_u == 'NW':
                            print(
                                f'Method={method_label} | Quarter={q} | '
                                f'bandwidth={nw_bandwidth:g} | eps={eps_value:.6g} | '
                                f'internal_CV_mean={cand_internal_cv_mean:.4f}'
                            )
                        else:
                            print(
                                f'Method={method_label} | Quarter={q} | eps={eps_value:.6g} | '
                                f'internal_CV_mean={cand_internal_cv_mean:.4f}'
                            )

                    candidate = {
                        'eps': eps_value,
                        'bandwidth': nw_bandwidth,
                        'internal_CV_mean': cand_internal_cv_mean,
                    }
                    if best is None or candidate['internal_CV_mean'] < best['internal_CV_mean']:
                        best = candidate
        finally:
            if cv_model_templates is not None:
                dispose_policy({'models': cv_model_templates})
            globals()['N'] = old_global_N

        time_eps_tuning_seconds = float(time.perf_counter() - t_eps_start)

        best_eps = best['eps']
        selected_nw_bandwidth = best['bandwidth'] if method_u == 'NW' else None
        internal_cv_mean = best['internal_CV_mean']
        internal_cv_train_n = int(IN_SAMPLE_CV_TRAIN_N)
        internal_cv_val_size = int(VAL_SIZE)

        if VERBOSE:
            if method_u == 'NW':
                print(
                    f'Method={method_label} | Quarter={q} selected '
                    f'bandwidth={selected_nw_bandwidth:g}, eps={best_eps:.6g} '
                    f'using internal_CV_mean={internal_cv_mean:.4f}'
                )
            else:
                print(
                    f'Method={method_label} | Quarter={q} selected eps={best_eps:.6g} '
                    f'using internal_CV_mean={internal_cv_mean:.4f}'
                )

        if method_u == 'NW':
            t0 = time.perf_counter()
            transition_matrix, transition_info = build_power_price_nw_transition_matrix(
                daily_power,
                daily_price_run,
                bandwidth=selected_nw_bandwidth,
            )
            transition_eval_mode = 'NW'
            time_transition_seconds += float(time.perf_counter() - t0)

        t0 = time.perf_counter()
        val_transition_cache = build_oos_transition_cache(
            transition_eval_mode,
            daily_wind_val,
            daily_power_val,
            daily_price_val_run,
            transition_info,
        )
        test_transition_cache = build_oos_transition_cache(
            transition_eval_mode,
            daily_wind_test,
            daily_power_test,
            daily_price_test_run,
            transition_info,
        )
        time_transition_seconds += float(time.perf_counter() - t0)

        if VERBOSE:
            print(
                f'Method={method_label} | Quarter={q} refitting selected hyperparameters '
                f'on full in-sample N={full_train_n} with MAX_ITER={int(MAX_ITER)}'
            )

        t0 = time.perf_counter()
        best_policy = solve_policy(
            daily_power,
            daily_price_run,
            transition_matrix,
            best_eps,
            num_iter=int(MAX_ITER),
            forward_uniforms=forward_uniforms,
            model_templates=base_model_templates,
        )
        time_sddp_solve_seconds += float(time.perf_counter() - t0)

        t0 = time.perf_counter()
        val_neg_raw = evaluate_oos(
            best_policy,
            daily_power,
            daily_price_run,
            daily_wind_val,
            daily_power_val,
            daily_price_val_run,
            daily_price_val,
            transition_info,
            transition_eval_mode,
            best_eps,
            transition_row_cache=val_transition_cache,
        )
        time_validation_eval_seconds += float(time.perf_counter() - t0)
        if val_neg_raw is None:
            dispose_policy(best_policy)
            raise ValueError('Validation split is empty; choose a positive VAL_SIZE with available holdout trajectories.')
        VAL_mean = float(np.mean(val_neg_raw))

        t0 = time.perf_counter()
        test_neg_raw = evaluate_oos(
            best_policy,
            daily_power,
            daily_price_run,
            daily_wind_test,
            daily_power_test,
            daily_price_test_run,
            daily_price_test,
            transition_info,
            transition_eval_mode,
            best_eps,
            transition_row_cache=test_transition_cache,
        )
        time_test_eval_seconds = float(time.perf_counter() - t0)
        if test_neg_raw is None:
            dispose_policy(best_policy)
            raise ValueError('Test split is empty after validation split; choose a smaller VAL_SIZE.')

    OOS_mean = float(np.mean(test_neg_raw))
    p10 = float(np.percentile(test_neg_raw, 10))
    p90 = float(np.percentile(test_neg_raw, 90))
    runtime_total_seconds = float(time.perf_counter() - runtime_start)

    if method_u == 'NN':
        print(
            f'Method=N-SDDP | Quarter={q} | VAL_mean={VAL_mean:.4f} | '
            f'OOS_mean={OOS_mean:.4f} | '
            f'total_with_teacher={time_nn_total_with_teacher_seconds:.2f} | '
            f'inference_excluding_teacher={time_nn_inference_excluding_teacher_seconds:.2f} | '
            f'reused_IND_teacher={nn_teacher_reused} | device={DEVICE.type}'
        )
    else:
        print(
            f'Method={method_label} | Quarter={q} | VAL_mean={VAL_mean:.4f} | '
            f'OOS_mean={OOS_mean:.4f} | '
            f'total_time={runtime_total_seconds:.2f} | solve_time={time_sddp_solve_seconds:.2f}'
        )

    best_eps_value = int(best_eps) if float(best_eps).is_integer() else float(best_eps)
    record = {
        'dataset': 'OH',
        'quarter': int(q),
        'method': method_u,
        'method_label': method_label,
        'N': int(N),
        'T': int(T),
        'VAL_SIZE': int(VAL_SIZE),
        'IN_SAMPLE_CV_TRAIN_N': internal_cv_train_n,
        'internal_CV_mean': internal_cv_mean,
        'NUM_ITER': int(NUM_ITER),
        'sddp_iterations_run': int(best_policy.get('iterations_run', NUM_ITER)),
        'selected_gmm_K': selected_gmm_K,
        'selected_gmm_reg_covar': selected_gmm_reg_covar,
        'selected_nw_kernel': 'exp_mahalanobis_product' if method_u == 'NW' else None,
        'selected_nw_bandwidth': selected_nw_bandwidth,
        'best_eps': best_eps_value,
        'VAL_mean': VAL_mean,
        'OOS_mean': OOS_mean,
        'p10': p10,
        'p90': p90,
        'neural_variant': neural_variant,
        'NUM_NEURAL_CUTS': int(NUM_NEURAL_CUTS) if neural_variant is not None else None,
        'NEURAL_REFINEMENT_ITER': int(NEURAL_REFINEMENT_ITER) if neural_variant is not None else None,
        'nn_train_device': DEVICE.type if neural_variant is not None else None,
        'nn_teacher_reused': bool(nn_teacher_reused) if neural_variant is not None else None,
        'runtime_total_seconds': runtime_total_seconds,
        'time_transition_seconds': time_transition_seconds,
        'time_eps_tuning_seconds': time_eps_tuning_seconds,
        'time_sddp_solve_seconds': time_sddp_solve_seconds,
        'time_VAL_eval_seconds': time_validation_eval_seconds,
        'time_validation_eval_seconds': time_validation_eval_seconds,
        'time_test_eval_seconds': time_test_eval_seconds,
        'time_nn_teacher_sddp_seconds': time_nn_teacher_sddp_seconds,
        'time_nn_training_seconds': time_nn_training_seconds,
        'time_nn_prediction_seconds': time_nn_prediction_seconds,
        'time_nn_final_forward_seconds': time_nn_final_forward_seconds,
        'time_nn_total_with_teacher_seconds': time_nn_total_with_teacher_seconds,
        'time_nn_inference_excluding_teacher_seconds': time_nn_inference_excluding_teacher_seconds,
    }
    dispose_policy(best_policy)
    return record

In [ ]:
# Main execution
outdir = Path.cwd() / 'Results'
outdir.mkdir(exist_ok=True)
summary_csv = outdir / 'OH_SDDP.csv'
if summary_csv.exists():
    summary_csv.unlink()

data_dir = find_wind_data_dir()

if VERBOSE:
    print('Using wind data directory:', data_dir)
mat_data = scipy.io.loadmat(data_dir / 'OH_all.mat')
mat_price = scipy.io.loadmat(data_dir / 'price_all.mat')

print('RUNNING METHODS:', METHODS)
allowed_methods = {"IND", "NW", "GMM", "NN"}
bad_methods = [m for m in METHODS if normalize_mode_name(m) not in allowed_methods]
assert not bad_methods, f"Unexpected METHODS: {bad_methods}"

all_records = []
for q in QUARTERS:
    weekly_wind = mat_data['weekly_wind'][0][q - 1] if 'weekly_wind' in mat_data else None
    weekly_power = mat_data['weekly_power'][0][q - 1]
    weekly_price = mat_price['weekly_price'][0][q - 1]

    t0_template = time.perf_counter()
    shared_model_templates = build_quarter_model_templates(weekly_power, weekly_price)
    if VERBOSE:
        print(
            f'Quarter={q} shared clean model templates built in '
            f'{time.perf_counter() - t0_template:.2f} seconds'
        )

    neural_teacher_cache = {}

    try:
        for method in METHODS:
            method_u = normalize_mode_name(method)
            forward_uniforms = make_forward_uniforms(BASE_SEED, q, NUM_ITER, T)
            record = run_quarter(
                method_u,
                q,
                weekly_wind,
                weekly_power,
                weekly_price,
                forward_uniforms,
                base_model_templates=shared_model_templates,
                neural_teacher_cache=neural_teacher_cache,
            )
            all_records.append(record)
    finally:
        dispose_policy({'models': shared_model_templates})

summary_fields = [
    'dataset',
    'quarter',
    'method',
    'method_label',
    'N',
    'T',
    'VAL_SIZE',
    'IN_SAMPLE_CV_TRAIN_N',
    'internal_CV_mean',
    'NUM_ITER',
    'sddp_iterations_run',
    'selected_gmm_K',
    'selected_gmm_reg_covar',
    'selected_nw_kernel',
    'selected_nw_bandwidth',
    'best_eps',
    'VAL_mean',
    'OOS_mean',
    'p10',
    'p90',
    'neural_variant',
    'NUM_NEURAL_CUTS',
    'NEURAL_REFINEMENT_ITER',
    'nn_train_device',
    'nn_teacher_reused',
    'runtime_total_seconds',
    'time_transition_seconds',
    'time_eps_tuning_seconds',
    'time_sddp_solve_seconds',
    'time_VAL_eval_seconds',
    'time_validation_eval_seconds',
    'time_test_eval_seconds',
    'time_nn_teacher_sddp_seconds',
    'time_nn_training_seconds',
    'time_nn_prediction_seconds',
    'time_nn_final_forward_seconds',
    'time_nn_total_with_teacher_seconds',
    'time_nn_inference_excluding_teacher_seconds',
]
with open(summary_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=summary_fields)
    writer.writeheader()
    writer.writerows(all_records)

if VERBOSE:
    print('\nSaved OH SDDP CSV:', summary_csv)